# MCP 서버 기초 — FastMCP 서버와 도구 정의

**Skilljar Lessons 01-04 대응**

이 노트북에서 다루는 내용:
1. MCP 개념과 3대 기능 (Tools, Resources, Prompts)
2. FastMCP 서버 생성
3. `@mcp.tool()` 데코레이터로 도구 정의
4. 자동 스키마 생성 확인

In [ ]:
# ── Setup ──────────────────────────────────────────────
# MCP SDK가 설치되어 있지 않으면 먼저 설치하세요:
# pip install "mcp[cli]"

import json
from datetime import datetime

## §1. MCP란? — Tool Use를 표준화하다

Week 04에서 배운 Tool Use는 도구 스키마와 실행 로직이 **앱 코드에 내장**되어 있었습니다.

MCP(Model Context Protocol)는 도구를 **독립된 서버**로 분리하여:
- 어떤 LLM 호스트든 같은 방식으로 연결 가능
- 한 번 만들어 여러 곳에서 재사용
- Tools + Resources + Prompts 3가지 기능 제공

| 항목 | Tool Use (W4) | MCP (W7) |
|------|--------------|----------|
| 도구 정의 위치 | 앱 코드 안에 JSON Schema | MCP 서버에 `@mcp.tool()` |
| 스키마 생성 | 수동 | 자동 (타입 힌트 + docstring) |
| 재사용성 | 해당 앱에서만 | 어떤 MCP 클라이언트든 |
| 추가 기능 | 도구만 | Tools + Resources + Prompts |

## §2. 최소 MCP 서버 생성

FastMCP를 사용하여 MCP 서버를 생성합니다. 아직 도구가 없는 빈 서버입니다.

In [ ]:
from mcp.server.fastmcp import FastMCP

# MCP 서버 인스턴스 생성
mcp = FastMCP("My First MCP Server")

print(f"서버 이름: {mcp.name}")
print("서버가 생성되었습니다. 아직 도구가 없으므로 기능은 비어 있습니다.")

## §3. @mcp.tool() 데코레이터로 도구 정의

Week 04에서는 JSON Schema를 수동으로 작성했지만, FastMCP에서는 **데코레이터 + 타입 힌트**만으로 충분합니다.

### Week 04 방식 (수동 스키마)
```python
# 1. 함수 작성
def get_weather(city: str) -> str: ...

# 2. JSON Schema 수동 작성 (번거롭다!)
schema = {
    "name": "get_weather",
    "description": "...",
    "input_schema": {"type": "object", "properties": {...}}
}

# 3. tools 배열에 등록
# 4. run_tool 라우터에 추가
```

### Week 07 방식 (FastMCP 자동 생성)
```python
@mcp.tool()
def get_weather(city: str) -> str:
    """도시의 날씨를 반환합니다."""
    ...
# 끝! 스키마 자동 생성, 라우팅 자동 처리
```

In [ ]:
# 도구 1: 현재 시간 조회
@mcp.tool()
def get_current_time(format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """현재 날짜와 시간을 반환합니다.

    Args:
        format: 날짜/시간 형식 (strftime 포맷)
    """
    return datetime.now().strftime(format)


# 도구 2: 두 숫자 더하기
@mcp.tool()
def add_numbers(a: float, b: float) -> float:
    """두 숫자를 더합니다.

    Args:
        a: 첫 번째 숫자
        b: 두 번째 숫자
    """
    return a + b


# 도구 3: 면적 계산
@mcp.tool()
def calculate_area(width: float, height: float, unit: str = "m") -> str:
    """직사각형 면적을 계산합니다.

    Args:
        width: 너비
        height: 높이
        unit: 단위 (기본값: m)
    """
    area = width * height
    return f"{area:.2f} {unit}²"


print("3개 도구가 등록되었습니다.")

## §4. 등록된 도구 확인

FastMCP에 등록된 도구 목록과 자동 생성된 스키마를 확인합니다.

In [ ]:
# 등록된 도구 목록 확인
import asyncio

async def list_tools():
    """등록된 도구 목록과 스키마를 출력합니다."""
    tools = await mcp.list_tools()
    print(f"등록된 도구 수: {len(tools)}\n")
    for tool in tools:
        print(f"--- {tool.name} ---")
        print(f"  설명: {tool.description}")
        print(f"  스키마: {json.dumps(tool.inputSchema, indent=4, ensure_ascii=False)}")
        print()

await list_tools()

## §5. 도구 직접 호출 테스트

서버에 등록된 도구를 프로그래밍 방식으로 직접 호출하여 테스트합니다.

In [ ]:
# 도구 직접 호출 테스트
async def test_tools():
    # 도구 1: 현재 시간
    result1 = await mcp.call_tool("get_current_time", {"format": "%Y-%m-%d %H:%M"})
    print(f"현재 시간: {result1}")

    # 도구 2: 덧셈
    result2 = await mcp.call_tool("add_numbers", {"a": 3.14, "b": 2.71})
    print(f"3.14 + 2.71 = {result2}")

    # 도구 3: 면적
    result3 = await mcp.call_tool("calculate_area", {"width": 300, "height": 600, "unit": "mm"})
    print(f"면적: {result3}")

await test_tools()

## §6. 서버를 파일로 저장

실제 MCP 서버는 독립 파일(`server.py`)로 작성하여 실행합니다.
아래 셀을 실행하면 `server.py` 파일이 생성됩니다.

In [ ]:
server_code = '''
from mcp.server.fastmcp import FastMCP
from datetime import datetime

mcp = FastMCP("My First MCP Server")


@mcp.tool()
def get_current_time(format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """현재 날짜와 시간을 반환합니다.

    Args:
        format: 날짜/시간 형식 (strftime 포맷)
    """
    return datetime.now().strftime(format)


@mcp.tool()
def add_numbers(a: float, b: float) -> float:
    """두 숫자를 더합니다.

    Args:
        a: 첫 번째 숫자
        b: 두 번째 숫자
    """
    return a + b


@mcp.tool()
def calculate_area(width: float, height: float, unit: str = "m") -> str:
    """직사각형 면적을 계산합니다.

    Args:
        width: 너비
        height: 높이
        unit: 단위 (기본값: m)
    """
    area = width * height
    return f"{area:.2f} {unit}²"


if __name__ == "__main__":
    mcp.run()
'''

with open("server.py", "w") as f:
    f.write(server_code.strip())

print("server.py 파일이 생성되었습니다.")
print("\n터미널에서 다음 명령으로 서버를 실행하세요:")
print("  python server.py")
print("\n또는 Inspector로 테스트:")
print("  mcp dev server.py")

## 핵심 정리

| 항목 | Week 04 (Tool Use) | Week 07 (MCP) |
|------|-------------------|---------------|
| 서버 생성 | 해당 없음 | `FastMCP("name")` |
| 도구 정의 | JSON Schema 수동 작성 | `@mcp.tool()` 데코레이터 |
| 스키마 | 딕셔너리로 직접 정의 | 타입 힌트 + docstring → 자동 생성 |
| 라우팅 | `run_tool()` 함수 직접 구현 | FastMCP가 자동 처리 |
| 테스트 | 코드에서 직접 호출 | MCP Inspector (브라우저 UI) |